In [ ]:
# Step 1: 挂载Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: 导入必要的库
import os
import shutil

# Step 3: 定义源路径和目标路径
# 假设当前脚本与train文件夹在同一目录下
# 源路径位于Google Drive中的'Colab Notebooks'文件夹
source_dir = '/content/drive/My Drive/Colab Notebooks'
public_tar = os.path.join(source_dir, 'public.tar.gz')
train_tar = os.path.join(source_dir, 'train.tar.gz')
model = os.path.join('/content/drive/My Drive/best (4)-swin-800img-last.pt')

# 目标路径设置为Colab的根目录/content
destination_dir = '/content'

# Step 4: 复制.tar.gz文件到Colab的本地目录
# 使用shutil.copy可以在Python中完成复制操作
shutil.copy(public_tar, destination_dir)
shutil.copy(train_tar, destination_dir)
shutil.copy(model, destination_dir)

# Step 5: 解压.tar.gz文件
# 使用tar命令解压到目标目录
!tar -xzvf /content/public.tar.gz -C /content/
!tar -xzvf /content/train.tar.gz -C /content/

# Step 6: （可选）删除.tar.gz文件以节省空间
# 如果您不再需要压缩文件，可以取消注释以下两行删除它们
os.remove('/content/public.tar.gz')
os.remove('/content/train.tar.gz')

# Step 7: 获取并打印当前工作目录路径
current_path = os.getcwd()
print(f"当前工作目录路径: {current_path}")

# Step 8: 列出当前目录的内容以验证解压是否成功
!ls -lh /content/


流式输出内容被截断，只能显示最后 5000 行内容。
train/images/train_5008.png
train/images/train_5009.png
train/images/train_5010.png
train/images/train_5011.png
train/images/train_5012.png
train/images/train_5013.png
train/images/train_5014.png
train/images/train_5015.png
train/images/train_5016.png
train/images/train_5017.png
train/images/train_5018.png
train/images/train_5019.png
train/images/train_5020.png
train/images/train_5021.png
train/images/train_5022.png
train/images/train_5023.png
train/images/train_5024.png
train/images/train_5025.png
train/images/train_5026.png
train/images/train_5027.png
train/images/train_5028.png
train/images/train_5029.png
train/images/train_5030.png
train/images/train_5031.png
train/images/train_5032.png
train/images/train_5033.png
train/images/train_5034.png
train/images/train_5035.png
train/images/train_5036.png
train/images/train_5037.png
train/images/train_5038.png
train/images/train_5039.png
train/images/train_5040.png
train/images/train_5041.png
train/images/train_50

In [ ]:
shutil.copy("/content/drive/My Drive/best (4)-swin-800img-last.pt", destination_dir)

'/content/best (4)-swin-800img-last.pt'

In [ ]:
shutil.copy("/content/ultralytics/runs/detect/exp_tamper_detection/weights/best-yolov8n验证.pt", source_dir)

In [ ]:

pt = os.path.join(source_dir, 'best.pt')
shutil.copy(pt,"/content", )

'/content/best.pt'

In [ ]:
shutil.copy('/content/ultralytics/runs/detect/exp_tamper_detection/weights/last.pt', '/content/drive/My Drive/Colab Notebooks')

FileNotFoundError: [Errno 2] No such file or directory: '/content/ultralytics/runs/detect/exp_tamper_detection/weights/last.pt'

In [ ]:

!pip install matplotlib tqdm opencv-python
!pip install scikit-learn

In [ ]:
import os
import json
import shutil
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import cv2
from sklearn.model_selection import train_test_split

In [ ]:
# 定义基本路径（请根据实际情况修改）
base_dir = '/content'

# 定义路径
images_dir = os.path.join(base_dir, 'train/images')
labels_json = os.path.join(base_dir, 'train/label_train.json')
labels_dir = os.path.join(base_dir, 'train/labels')

# 创建 labels 目录
os.makedirs(labels_dir, exist_ok=True)

In [ ]:
# 加载 JSON 文件
with open(labels_json, 'r') as f:
    labels_data = json.load(f)

# 转换标签为 YOLO 格式
for idx, item in tqdm(enumerate(labels_data), total=len(labels_data)):
    image_id = item['id']
    regions = item['region']
    image_path = os.path.join(images_dir, image_id)
    img = cv2.imread(image_path)
    if img is None:
        print(f"Image not found: {image_path}")
        continue
    img_height, img_width = img.shape[:2]

    label_file = os.path.join(labels_dir, image_id.replace('.jpg', '.txt'))

    with open(label_file, 'w') as f:
        for box in regions:
            x_min, y_min, x_max, y_max = box
            # 确保坐标在图像范围内
            x_min = max(0, min(x_min, img_width - 1))
            x_max = max(0, min(x_max, img_width - 1))
            y_min = max(0, min(y_min, img_height - 1))
            y_max = max(0, min(y_max, img_height - 1))

            # 计算 YOLO 格式的坐标
            x_center = (x_min + x_max) / 2 / img_width
            y_center = (y_min + y_max) / 2 / img_height
            width = (x_max - x_min) / img_width
            height = (y_max - y_min) / img_height

            # YOLO 格式：class x_center y_center width height
            f.write(f"0 {x_center} {y_center} {width} {height}\n")

100%|██████████| 13000/13000 [00:58<00:00, 221.71it/s]


In [ ]:
# 获取所有图像文件的列表
image_files = [f for f in os.listdir(images_dir) if f.endswith('.jpg')]

# 拆分为训练集和验证集
train_files, val_files = train_test_split(image_files, test_size=0.2, random_state=42)

# 创建数据集目录
dataset_dir = os.path.join(base_dir, 'dataset')
train_images_dir = os.path.join(dataset_dir, 'images/train')
val_images_dir = os.path.join(dataset_dir, 'images/val')
train_labels_dir = os.path.join(dataset_dir, 'labels/train')
val_labels_dir = os.path.join(dataset_dir, 'labels/val')

os.makedirs(train_images_dir, exist_ok=True)
os.makedirs(val_images_dir, exist_ok=True)
os.makedirs(train_labels_dir, exist_ok=True)
os.makedirs(val_labels_dir, exist_ok=True)

# 复制文件到相应的目录
for file_list, images_dest, labels_dest in [
    (train_files, train_images_dir, train_labels_dir),
    (val_files, val_images_dir, val_labels_dir)
]:
    for filename in tqdm(file_list):
        # 复制图像
        shutil.copy(os.path.join(images_dir, filename), os.path.join(images_dest, filename))
        # 复制标签
        label_src = os.path.join(labels_dir, filename.replace('.jpg', '.txt'))
        label_dest = os.path.join(labels_dest, filename.replace('.jpg', '.txt'))
        if os.path.exists(label_src):
            shutil.copy(label_src, label_dest)
        else:
            # 如果标签文件不存在，创建一个空的标签文件
            open(label_dest, 'w').close()

100%|██████████| 1496/1496 [00:00<00:00, 2693.80it/s]


In [ ]:
# 生成包含数据增强参数的 data.yaml
data_yaml = f"""
train: {train_images_dir}
val: {val_images_dir}

nc: 1
names: ['tampered_region']

# 数据增强参数
lr0: 0.01  # 初始学习率
momentum: 0.937
weight_decay: 0.0005

degrees: 10.0        # 旋转角度范围（±10度）
translate: 0.1       # 平移范围（±10%）
scale: {0.8, 1.5}  # 图像随机缩放范围（1.5倍到3倍）           # 缩放范围（0.5到1.5倍）
shear: 2.0           # 剪切范围（±2度）
perspective: 0.0     # 透视变换概率
flipud: 0.05          # 上下翻转概率
fliplr: 0.5          # 左右翻转概率
mixup: 0.2           # MixUp 数据增强的概率
mosaic: 1.0          # Mosaic 数据增强的概率
"""

# 写入 data.yaml 文件
data_yaml_path = os.path.join(base_dir, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml)

In [ ]:
!rm -rf /content/ultralytics
!git clone https://github.com/ultralytics/ultralytics.git
!ls /content/ultralytics
!pip install -e /content/ultralytics

Cloning into 'ultralytics'...
remote: Enumerating objects: 45489, done.
remote: Counting objects: 100% (754/754), done.
remote: Compressing objects: 100% (470/470), done.
remote: Total 45489 (delta 493), reused 458 (delta 279), pack-reused 44735 (from 1)
Receiving objects: 100% (45489/45489), 38.74 MiB | 34.49 MiB/s, done.
Resolving deltas: 100% (33746/33746), done.
CITATION.cff	 docker  examples  mkdocs.yml	   README.md	    tests
CONTRIBUTING.md  docs	 LICENSE   pyproject.toml  README.zh-CN.md  ultralytics
Obtaining file:///content/ultralytics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.49-0.editable-py3-none-any.whl size=22709 sha256=ce2a21d6e318ec638968556d37899e6ac055b915ee610ed67ebdfab83fd81701
  Store

In [ ]:
!yolo train \
    model=yolov8n.pt \
    data=/content/data.yaml \
    epochs=200 \
    imgsz=512 \
    batch=192 \
    name=exp_tamper_detection \
    verbose=False \
    save_period=1 \
    lr0=1e-4 \
    optimizer='AdamW'

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
100% 6.25M/6.25M [00:00<00:00, 60.1MB/s]
Ultralytics 8.3.49 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/data.yaml, epochs=200, time=None, patience=100, batch=192, imgsz=512, save=True, save_period=1, cache=False, device=None, workers=8, project=None, name=exp_tamper_detection, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=False, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=Fal

In [ ]:
!pip install timm

In [ ]:
from ultralytics import YOLO  # 导入 YOLO 类

# 初始化 YOLOv8 模型（加载自定义配置文件）
model = YOLO("/content/best-2.pt")  # 使用自定义的 yaml 文件初始化模型
# 设置训练参数
# data_yaml_path = "ultralytics/cfg/default.yaml"  # 数据集配置文件路径
epochs = 120  # 自定义训练的轮数
img_size = 640  # 输入图像尺寸
batch_size = 64  # 批量大小，根据显存调整
exp_name = 'exp_tamper_detection'  # 实验名称

# 开始训练
model.train(
    data=data_yaml_path,  # 数据集配置文件路径
    epochs=epochs,  # 训练轮数
    imgsz=img_size,  # 输入图像尺寸
    batch=batch_size,  # 批量大小
    name=exp_name,  # 实验名称
    verbose=True,  # 是否输出详细信息
)

ImportError: cannot import name 'YOLO' from 'ultralytics' (unknown location)

In [ ]:
# 定义保存预测结果的目录
predictions_dir = os.path.join(base_dir, 'predictions')
os.makedirs(predictions_dir, exist_ok=True)

In [ ]:
model.save("/gaobudongle.pt")  # 替换为你想保存的路径

In [ ]:
# 获取验证集图像文件列表
val_image_files = [f for f in os.listdir(val_images_dir) if f.endswith('.jpg')]

# 存储预测结果
results_list = []

for filename in tqdm(val_image_files):
    image_path = os.path.join(val_images_dir, filename)
    # 读取图像
    img = cv2.imread(image_path)
    if img is None:
        print(f"Image not found: {image_path}")
        continue
    # 进行预测
    results = model.predict(image_path)
    # 获取预测的边界框（xyxy 格式）
    boxes = results[0].boxes.xyxy.cpu().numpy()
    # 将边界框转换为列表，并保留四个坐标值
    boxes_list = boxes.tolist()
    # 构建结果字典
    result_item = {
        'id': filename,
        'region': boxes_list  # 这里的 boxes_list 是一个二维列表，包含多个边界框
    }
    results_list.append(result_item)

    # 可视化预测结果并保存
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    # 保存带有预测框的图像
    save_path = os.path.join(predictions_dir, filename)
    cv2.imwrite(save_path, img)

  0%|          | 0/1496 [00:00<?, ?it/s]


image 1/1 /content/dataset/images/val/train_13128.jpg: 224x640 (no detections), 65.8ms
Speed: 1.7ms preprocess, 65.8ms inference, 0.9ms postprocess per image at shape (1, 3, 224, 640)


  0%|          | 1/1496 [00:00<06:23,  3.90it/s]


image 1/1 /content/dataset/images/val/train_7018.jpg: 640x640 (no detections), 12.5ms
Speed: 3.1ms preprocess, 12.5ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /content/dataset/images/val/train_8015.jpg: 224x640 (no detections), 13.1ms
Speed: 1.4ms preprocess, 13.1ms inference, 0.6ms postprocess per image at shape (1, 3, 224, 640)

image 1/1 /content/dataset/images/val/train_13421.jpg: 256x640 (no detections), 59.8ms
Speed: 1.6ms preprocess, 59.8ms inference, 0.7ms postprocess per image at shape (1, 3, 256, 640)


  0%|          | 4/1496 [00:00<02:14, 11.08it/s]

...


image 1/1 /content/dataset/images/val/train_8061.jpg: 288x640 (no detections), 14.3ms
Speed: 1.9ms preprocess, 14.3ms inference, 0.8ms postprocess per image at shape (1, 3, 288, 640)

image 1/1 /content/dataset/images/val/train_2102.jpg: 640x480 (no detections), 17.2ms
Speed: 2.8ms preprocess, 17.2ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /content/dataset/images/val/train_1656.jpg: 640x480 (no detections), 11.2ms
Speed: 3.1ms preprocess, 11.2ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 480)


 16%|█▌        | 240/1496 [00:11<00:59, 21.19it/s]


image 1/1 /content/dataset/images/val/train_1056.jpg: 640x480 (no detections), 12.0ms
Speed: 2.4ms preprocess, 12.0ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /content/dataset/images/val/train_9376.jpg: 640x448 (no detections), 11.8ms
Speed: 2.3ms preprocess, 11.8ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 448)



 16%|█▌        | 242/1496 [00:11<00:58, 21.56it/s]


KeyboardInterrupt: 

In [ ]:
# 定义结果文件路径
results_json_path = os.path.join(base_dir, 'predictions.json')

# 保存结果到 JSON 文件
with open(results_json_path, 'w') as f:
    json.dump(results_list, f, ensure_ascii=False, indent=4)

In [ ]:
# 6.1 导入必要的库
import numpy as np

# 6.2 定义计算 IoU 的函数
def compute_iou(box1, box2):
    x1_max = max(box1[0], box2[0])
    y1_max = max(box1[1], box2[1])
    x2_min = min(box1[2], box2[2])
    y2_min = min(box1[3], box2[3])

    inter_width = x2_min - x1_max
    inter_height = y2_min - y1_max
    if inter_width <= 0 or inter_height <= 0:
        return 0.0  # 没有重叠区域
    inter_area = inter_width * inter_height

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union_area = area1 + area2 - inter_area

    iou = inter_area / union_area
    return iou

# 6.3 加载真实标签和预测结果
# 加载真实标签
ground_truth = {}
for filename in val_image_files:
    label_file = os.path.join(val_labels_dir, filename.replace('.jpg', '.txt'))
    regions = []
    if os.path.exists(label_file):
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                x_center = float(parts[1])
                y_center = float(parts[2])
                width = float(parts[3])
                height = float(parts[4])

                # 转换为像素坐标
                image_path = os.path.join(val_images_dir, filename)
                img = cv2.imread(image_path)
                img_height, img_width = img.shape[:2]

                x_center *= img_width
                y_center *= img_height
                width *= img_width
                height *= img_height

                x_min = x_center - width / 2
                y_min = y_center - height / 2
                x_max = x_center + width / 2
                y_max = y_center + height / 2

                regions.append([x_min, y_min, x_max, y_max])
    ground_truth[filename] = regions

# 加载预测结果
predictions = {}
for item in results_list:
    filename = item['id']
    boxes = item['region']
    predictions[filename] = boxes

# 6.4 计算 TP、FP、FN
iou_threshold = 0.5

TP = 0
FP = 0
FN = 0

for filename in val_image_files:
    gt_boxes = ground_truth.get(filename, [])
    pred_boxes = predictions.get(filename, [])

    matched = []
    for pred_box in pred_boxes:
        pred_matched = False
        for gt_box in gt_boxes:
            iou = compute_iou(pred_box, gt_box)
            if iou >= iou_threshold:
                TP += 1
                pred_matched = True
                matched.append(gt_box)
                break
        if not pred_matched:
            FP += 1
    unmatched_gt = [gt for gt in gt_boxes if gt not in matched]
    FN += len(unmatched_gt)

# 6.5 计算 Precision、Recall 和 Micro-F1
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

if precision + recall == 0:
    micro_f1 = 0
else:
    micro_f1 = 2 * precision * recall / (precision + recall)

print(f"True Positive (TP): {TP}")
print(f"False Positive (FP): {FP}")
print(f"False Negative (FN): {FN}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Micro-F1 Score: {micro_f1:.4f}")

True Positive (TP): 584
False Positive (FP): 67
False Negative (FN): 190
Precision: 0.8971
Recall: 0.7545
Micro-F1 Score: 0.8196


In [ ]:
!unzip -p /content/drive/MyDrive/public.tar.gz.zip

In [ ]:
import os
import json
from tqdm import tqdm

# 6. 执行预测命令，保存 JSON 和 TXT 结果

# 7. 后处理 JSON 结果
input_json_path = '/content/ultralytics/runs/detect/exp_tamper_detection_predict/predict.json'
output_json_path = '/content/CV小队Pro_result.json'

# 检查输入文件是否存在
if not os.path.exists(input_json_path):
    print(f"错误: 输入 JSON 文件不存在: {input_json_path}")
    image_files = []
else:
    # 读取 COCO 格式的 JSON 文件
    with open(input_json_path, 'r', encoding='utf-8') as f:
        coco_data = json.load(f)

    # 创建一个字典，按图片ID存储
    image_id_to_file = {image['id']: image['file_name'] for image in coco_data['images']}

    # 初始化结果列表
    results = []

    # 遍历每个检测
    for annotation in tqdm(coco_data['annotations'], desc="Processing Annotations"):
        image_id = annotation['image_id']
        file_name = image_id_to_file.get(image_id, 'unknown')
        bbox = annotation['bbox']  # [x, y, width, height]

        # 将 bbox 转换为 [x_min, y_min, x_max, y_max]
        x_min, y_min, width, height = bbox
        x_max = x_min + width
        y_max = y_min + height
        region = [round(x_min, 2), round(y_min, 2), round(x_max, 2), round(y_max, 2)]

        # 查找或创建该图片的 entry
        existing_entry = next((item for item in results if item["id"] == file_name), None)
        if existing_entry is None:
            results.append({"id": file_name, "region": [region]})
        else:
            existing_entry["region"].append(region)

    # 写入自定义 JSON 文件
    with open(output_json_path, 'w', encoding='utf-8') as f:
        for entry in results:
            json_line = json.dumps(entry, ensure_ascii=False)
            f.write(json_line + '\n')

    print(f"自定义结果文件已保存至: {output_json_path}")

Ultralytics 8.3.40 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
Model summary (fused): 268 layers, 68,124,531 parameters, 0 gradients, 257.4 GFLOPs

image 1/1200 /content/val_images/val_1000.png: 640x640 (no detections), 82.7ms
image 2/1200 /content/val_images/val_1001.png: 640x640 (no detections), 55.6ms
image 3/1200 /content/val_images/val_1002.png: 640x640 (no detections), 55.0ms
image 4/1200 /content/val_images/val_1003.png: 640x640 (no detections), 51.6ms
image 5/1200 /content/val_images/val_1004.png: 640x640 (no detections), 49.6ms
image 6/1200 /content/val_images/val_1005.png: 640x640 (no detections), 51.2ms
image 7/1200 /content/val_images/val_1006.png: 640x640 (no detections), 51.2ms
image 8/1200 /content/val_images/val_1007.png: 640x640 (no detections), 52.8ms
image 9/1200 /content/val_images/val_1008.png: 640x640 (no detections), 50.3ms
image 10/1200 /content/val_images/val_1009.png: 640x640 (no detections), 50.4ms
image 11/1200 /content/val_images/val_1010

In [ ]:
import os
import json
from tqdm import tqdm
from PIL import Image

# 定义基本路径
base_dir = '/content'  # 根据实际情况修改
val_images_dir = os.path.join(base_dir, 'val_images')
labels_dir = os.path.join(base_dir, 'ultralytics', 'runs', 'detect', 'exp_tamper_detection_predict', 'labels')
output_filename = 'CV小队Pro_result.json'
output_path = os.path.join(base_dir, output_filename)

# 定义标签文件的扩展名
label_extension = '.txt'

# 获取验证集中的所有标签文件
try:
    label_files = [f for f in os.listdir(labels_dir) if f.lower().endswith(label_extension)]
except FileNotFoundError:
    print(f"错误: 标签目录不存在: {labels_dir}")
    label_files = []

print(f"验证集标签文件数量: {len(label_files)}")

# 初始化结果列表
results = []

for label_file in tqdm(label_files, desc="Processing Label Files"):
    # 获取对应的图片文件名
    image_filename = label_file.replace(label_extension, '.jpg')  # 假设图片为.jpg格式
    image_path_jpg = os.path.join(val_images_dir, image_filename)
    if not os.path.exists(image_path_jpg):
        image_filename = label_file.replace(label_extension, '.png')  # 尝试.png格式
        image_path = os.path.join(val_images_dir, image_filename)
        if not os.path.exists(image_path):
            print(f"警告: 找不到对应的图像文件 for {label_file}")
            continue
    else:
        image_path = image_path_jpg

    # 读取图像以获取宽度和高度
    try:
        with Image.open(image_path) as img:
            width, height = img.size
    except Exception as e:
        print(f"错误: 无法打开图像文件: {image_path}")
        print(f"错误信息: {e}")
        continue

    # 读取标签文件并解析
    regions = []
    label_path = os.path.join(labels_dir, label_file)
    try:
        with open(label_path, 'r') as lf:
            lines = lf.readlines()
    except Exception as e:
        print(f"错误: 无法读取标签文件: {label_path}")
        print(f"错误信息: {e}")
        continue

    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            print(f"警告: 无效的标签格式 in {label_path}: {line}")
            continue
        class_id, x_center, y_center, bbox_width, bbox_height = parts
        try:
            x_center = float(x_center)
            y_center = float(y_center)
            bbox_width = float(bbox_width)
            bbox_height = float(bbox_height)
        except ValueError:
            print(f"警告: 无效的数值 in {label_path}: {line}")
            continue

        # 将归一化坐标转换为像素坐标
        x_min = (x_center - bbox_width / 2) * width
        y_min = (y_center - bbox_height / 2) * height
        x_max = (x_center + bbox_width / 2) * width
        y_max = (y_center + bbox_height / 2) * height

        # 四舍五入为两位小数
        x_min = round(x_min, 2)
        y_min = round(y_min, 2)
        x_max = round(x_max, 2)
        y_max = round(y_max, 2)

        regions.append([x_min, y_min, x_max, y_max])

    # 创建 JSON 对象
    data = {
        "id": image_filename,
        "region": regions
    }
    results.append(data)

# 将结果写入 JSON 文件
try:
    with open(output_path, 'w', encoding='utf-8') as outfile:
        for entry in results:
            json_line = json.dumps(entry, ensure_ascii=False)
            outfile.write(json_line + '\n')
    print(f"结果文件已保存至: {output_path}")
except Exception as e:
    print(f"错误: 无法写入输出文件: {output_path}")
    print(f"错误信息: {e}")

验证集标签文件数量: 499


Processing Label Files: 100%|██████████| 499/499 [00:00<00:00, 5805.21it/s]

警告: 找不到对应的图像文件 for val_1233.txt
警告: 找不到对应的图像文件 for val_1477.txt
警告: 找不到对应的图像文件 for val_1418.txt
警告: 找不到对应的图像文件 for val_1463.txt
警告: 找不到对应的图像文件 for val_1482.txt
结果文件已保存至: /content/CV小队Pro_result.json


In [ ]:
import os
import json
from tqdm import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt

# 如果需要可视化检测结果，可以取消以下导入的注释
# import cv2  # 需要安装 OpenCV: pip install opencv-python

# 定义基本路径
base_dir = '/content'  # 根据实际情况修改

# 定义验证集图片路径
val_images_dir = os.path.join(base_dir, 'val_images')

# 定义输出结果文件名（请将 'YourTeamName' 替换为您的实际团队名称）
output_filename = 'CV小队Pro_result.json'
output_path = os.path.join(base_dir, output_filename)

# 获取验证集中的所有图片文件
image_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
try:
    image_files = [f for f in os.listdir(val_images_dir) if f.lower().endswith(image_extensions)]
except FileNotFoundError:
    print(f"错误: 验证集图片目录不存在: {val_images_dir}")
    image_files = []

print(f"验证集图片数量: {len(image_files)}")

# 加载 YOLO 模型
# 请根据您的实际模型路径或模型名称进行调整
# 例如，如果使用自定义训练的模型，请提供相应的路径
model_path = 'content/best (1).pt'  # 替换为您实际使用的模型路径
try:
    model = YOLO(model_path)
    print(f"成功加载模型: {model_path}")
except Exception as e:
    print(f"加载模型失败: {model_path}")
    print(f"错误信息: {e}")
    exit(1)

# 打开输出文件，准备写入结果
with open(output_path, 'w', encoding='utf-8') as outfile:
    # 使用 tqdm 显示进度条
    for image_file in tqdm(image_files, desc="Processing Images"):
        image_path = os.path.join(val_images_dir, image_file)

        # 使用模型进行推理，设置适当的置信度阈值
        try:
            results = model(image_path, conf=0.25)  # 可以调整 conf 参数
        except Exception as e:
            print(f"推理失败: {image_path}")
            print(f"错误信息: {e}")
            continue  # 跳过此图片，继续处理下一个

        # 初始化该图片的 region 列表
        regions = []

        # 解析推理结果
        for result in results:
            # 获取所有检测到的框
            boxes = result.boxes  # Boxes object

            # 遍历每个检测框
            for box in boxes:
                # 获取边界框的坐标 (x_min, y_min, x_max, y_max)
                # xyxy 属性返回的是 tensor，需要转换为列表
                xyxy = box.xyxy[0].tolist()  # [x_min, y_min, x_max, y_max]

                # 将坐标四舍五入为两位小数（根据需要可以保留小数）
                xyxy = [round(coord, 2) for coord in xyxy]

                # 添加到 regions 列表
                regions.append(xyxy)

        # 创建 JSON 对象
        data = {
            "id": image_file,
            "region": regions
        }

        # 将 JSON 对象写入文件，确保中文字符不被转义
        json_line = json.dumps(data, ensure_ascii=False)
        outfile.write(json_line + '\n')

        # 打印当前图片的所有 regions
        print(f"Image: {image_file}")
        if regions:
            for idx, region in enumerate(regions, start=1):
                print(f"  Region {idx}: {region}")
        else:
            print("  No detections found.")
        print("-" * 50)  # 分隔线，便于阅读

        # 可视化检测结果（可选）
        # 如果需要可视化检测结果，请取消以下代码块的注释

print(f"结果文件已保存至: {output_path}")


ImportError: cannot import name 'YOLO' from 'ultralytics' (unknown location)